# 0.2 - Main data preparation
Generating models main input data from datasources:
- Brazilian E-Commerce Public Dataset by Olist
- MNIST image number double digit custom dataset

## 1: Dependencies and Imports

In [1]:
%pip install kagglehub pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Standard library
import os
import random

# Third-party
import kagglehub
import pandas as pd
import torch

## 2: Load Original Data

In [3]:
# Download and access the dataset path
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

# List of filenames to be processed
filenames = [
    'olist_customers_dataset.csv', 'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv',
    'olist_orders_dataset.csv', 'olist_products_dataset.csv',
    'olist_sellers_dataset.csv', 'product_category_name_translation.csv'
]

# Automated data loading into a dictionary
datasets = {}
for file in filenames:
    # Clean the filename to create a concise dictionary key
    key = file.replace('.csv', '').replace('olist_', '').replace('_dataset', '')
    datasets[key] = pd.read_csv(os.path.join(path, file))
    
    print(f"Loaded: {key} ({datasets[key].shape[0]} rows)")

Loaded: customers (99441 rows)
Loaded: order_items (112650 rows)
Loaded: order_payments (103886 rows)
Loaded: order_reviews (99224 rows)
Loaded: orders (99441 rows)
Loaded: products (32951 rows)
Loaded: sellers (3095 rows)
Loaded: product_category_name_translation (71 rows)


In [4]:
# Load the pre-generated datasets from disk
train_data = torch.load('../data/MNIST_2DIGITS/train_2digits.pt')
test_data = torch.load('../data/MNIST_2DIGITS/test_2digits.pt')

# Combine all samples to maximize the available data pool
all_images = torch.cat((train_data['images'], test_data['images']), dim=0)
all_labels = torch.cat((train_data['labels'], test_data['labels']), dim=0)

print(f"Total two-digit images available: {len(all_labels)}\n")

Total two-digit images available: 70000



## 3: Data combination and pre-integration

In [5]:
# 1. Merge Orders and Customers
df = pd.merge(datasets['orders'], datasets['customers'], on='customer_id', how='left')

# 2. Add Order Items (Note: This increases row count due to multiple items per order)
df = pd.merge(df, datasets['order_items'], on='order_id', how='left')

# 3. Add Products and Category Translations
df = pd.merge(df, datasets['products'], on='product_id', how='left')
df = pd.merge(df, datasets['product_category_name_translation'], on='product_category_name',  how='left')

# 4. Add Sellers
df = pd.merge(df, datasets['sellers'], on='seller_id', how='left')

# 5. Add Payments and Reviews
# Merging payments and reviews directly may create duplicates if an order has multiple entries.
df = pd.merge(df, datasets['order_payments'], on='order_id', how='left')
df = pd.merge(df, datasets['order_reviews'], on='order_id', how='left')

print(f"\nFinal integrated dataset. Total rows: {df.shape[0]}")


Final integrated dataset. Total rows: 119143


In [6]:
# (Brazilian) State Mapping to numeric values (0-26)

# Get unique states and create mapping dictionaries
unique_states = sorted(df['customer_state'].dropna().unique())
state_to_idx = {state: i for i, state in enumerate(unique_states)}
idx_to_state = {i: state for state, i in state_to_idx.items()}

# Map states to numbers; fill missing values with -1
df['customer_state_num'] = df['customer_state'].map(state_to_idx).fillna(-1).astype(int)
df['seller_state_num'] = df['seller_state'].map(state_to_idx).fillna(-1).astype(int)

# Drop original categorical state columns
df.drop(columns=['customer_state', 'seller_state'], inplace=True)

print("State mapping completed (0-26).\n")

State mapping completed (0-26).



In [7]:
# Create pools of images for each state based on the mapped numeric values
state_images_pool = {}

print("Creating pools using all available variations per state...")

# Iterate through state indices (0-26) from the mapping
for state_idx in state_to_idx.values():
    # Find all indices in all_labels that match the current state
    matching_indices = (all_labels == state_idx).nonzero(as_tuple=True)[0]
    
    if len(matching_indices) > 0:
        # Store all available images for this state without sampling limits
        state_images_pool[state_idx] = all_images[matching_indices]
    else:
        print(f"Warning: No images found in MNIST for state index {state_idx}")

Creating pools using all available variations per state...


In [8]:
# Random assigment of images to orders based on state mapping

# Reference the original image dimensions
image_shape = all_images[0].shape

def get_random_image(state_idx):
    """Retrieves a random image from the complete pool for a given state index."""
    if state_idx == -1 or state_idx not in state_images_pool:
        return None 
    
    # Select a random image from the full list of available samples
    pool = state_images_pool[state_idx]
    random_idx = random.randint(0, len(pool) - 1)
    return pool[random_idx]

print("Assigning unique calligraphy to each order...")

# Apply the function to map state numbers to images
df['customer_state_image'] = df['customer_state_num'].apply(get_random_image)
df['seller_state_image'] = df['seller_state_num'].apply(get_random_image)

Assigning unique calligraphy to each order...


## 4: Saving Data

In [9]:
# Tabular data save in CSV

# Path definition
os.makedirs('../data/raw_input', exist_ok=True)

# save except the image columns
tabular_data = df.drop(columns=['customer_state_image', 'seller_state_image'])
tabular_data.to_csv('../data/raw_input/m_tabular.csv', index=False)
print("1/3: m_tabular.csv saved.")

1/3: m_tabular.csv saved.


In [10]:
def save_visual_dataset(df, img_column, filename):
    """Saves order IDs and image tensors to a PyTorch file with memory optimization."""
    order_ids = df['order_id'].tolist()
    
    # Handle missing images by substituting them with a black tensor
    images = [img if img is not None else torch.zeros(image_shape) for img in df[img_column]]
    
    # Convert float tensors to uint8 to reduce file size by approximately 75%
    uint8_tensor = (torch.stack(images) * 255).to(torch.uint8)
    
    save_path = f'../data/raw_input/{filename}'
    torch.save({
        'order_id': order_ids,
        'images': uint8_tensor
    }, save_path)
    
    print(f"Dataset saved: {filename}")

In [11]:
# Customer (Brazilian) state images (Compressed .pt)
save_visual_dataset(df, 'customer_state_image', 'm_cust_images.pt')
print("2/3: m_cust_images.pt saved (Uint8 format).")

Dataset saved: m_cust_images.pt
2/3: m_cust_images.pt saved (Uint8 format).


In [12]:
# Vendor (Brazilian) state images (Compressed .pt)
save_visual_dataset(df, 'seller_state_image', 'm_sell_images.pt')
print("3/3: m_sell_images.pt saved (Uint8 format).")

Dataset saved: m_sell_images.pt
3/3: m_sell_images.pt saved (Uint8 format).
